# 01 Exploration — MP Theory and Single-Window Spectral Analysis

This notebook walks through the core theoretical objects interactively:

1. Marčenko-Pastur density for various γ and σ²
2. BBP threshold vs. bulk edge distinction (Mathematical Flag 1)
3. Self-consistent σ² estimator convergence
4. Single-window spectral analysis on synthetic data
5. `verify_bbp_distinction` cross-check

This notebook is for exploration only — no real data required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

from research.core.mp_theory import (
    mp_density, mp_cdf, bulk_edges, bbp_threshold,
    bbp_sample_eigenvalue, estimate_sigma2_self_consistent,
    verify_bbp_distinction, ks_distance_from_mp,
)
from research.core.estimator import RollingSpectralEstimator

rng = np.random.default_rng(42)

## 1. MP Density — Varying γ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=False)
sigma2 = 1.0

for ax, gamma in zip(axes, [0.1, 0.5, 0.9]):
    lm, lp = bulk_edges(gamma, sigma2)
    x = np.linspace(max(lm * 0.5, 1e-6), lp * 1.2, 600)
    y = [mp_density(xi, gamma, sigma2) for xi in x]
    ax.plot(x, y, lw=1.5)
    ax.axvline(lp, color='red', ls='--', lw=0.9, label=r'$\lambda_+$')
    if lm > 1e-6:
        ax.axvline(lm, color='grey', ls=':', lw=0.9, label=r'$\lambda_-$')
    ax.set_title(f'$\\gamma={gamma}$, $\\sigma^2={sigma2}$')
    ax.set_xlabel('Eigenvalue')
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle('Marčenko-Pastur density for varying aspect ratio γ', y=1.02)
plt.tight_layout()
plt.show()

## 2. Mathematical Flag 1: θ_c ≠ λ+

> **Flag 1**: The BBP detection threshold θ_c = σ²(1+√γ) and the MP bulk edge λ+ = σ²(1+√γ)² are **distinct** quantities. The ratio λ+/θ_c = 1+√γ > 1 always.

In [ ]:
# Analytical verification for gamma=0.25, sigma2=1.0
gamma, sigma2 = 0.25, 1.0
theta_c = bbp_threshold(gamma, sigma2)
_, lp = bulk_edges(gamma, sigma2)

print(f'gamma={gamma}, sigma2={sigma2}')
print(f'  theta_c = sigma2*(1+sqrt(gamma)) = {theta_c:.6f}  (expected: 1.500000)')
print(f'  lambda+ = sigma2*(1+sqrt(gamma))^2 = {lp:.6f}  (expected: 2.250000)')
print(f'  ratio lambda+/theta_c = {lp/theta_c:.6f}  (expected: 1.500000)')
print()

# Interactive cross-check
verify_bbp_distinction(gamma=0.25, sigma2=1.0)

In [ ]:
# Show ratio across gamma values
gammas = np.linspace(0.01, 0.99, 200)
ratios = [(1 + np.sqrt(g)) for g in gammas]  # lambda+/theta_c = 1+sqrt(gamma)

plt.figure(figsize=(5, 3))
plt.plot(gammas, ratios)
plt.axhline(1.0, color='grey', ls='--', lw=0.8)
plt.xlabel(r'$\gamma = d/T$')
plt.ylabel(r'$\lambda_+ / \theta_c$')
plt.title(r'$\lambda_+/\theta_c = 1+\sqrt{\gamma}$ — always > 1 (Flag 1)')
plt.tight_layout()
plt.show()

## 3. Self-Consistent σ² — Convergence

In [ ]:
# Pure noise: gamma=0.3, sigma2_true=1.0
# Generate eigenvalues from a Wishart(I, d, T) matrix
gamma_test = 0.3
sigma2_true = 1.0
d, T = 150, 500

X = rng.standard_normal((T, d))
S = X.T @ X / T
eigs = np.sort(np.linalg.eigvalsh(S))[::-1]

sigma2_est, n_iter, converged = estimate_sigma2_self_consistent(eigs, d/T)
print(f'True sigma2: {sigma2_true:.4f}')
print(f'Naive mean of all eigs: {np.mean(eigs):.4f}')
print(f'Self-consistent estimate: {sigma2_est:.4f}  (converged={converged}, iters={n_iter})')

## 4. Single-Window Analysis — Sector Factor Model

In [ ]:
# Sector factor model: 3 sectors, alpha=0.6
# Expected spike eigenvalue: 1 + (d/3 - 1) * 0.36
d, T, n_sectors, alpha = 60, 500, 3, 0.6
m = d // n_sectors  # assets per sector

# Generate sector factor returns
factors = rng.standard_normal((T, n_sectors))
idiosync = rng.standard_normal((T, d))
X = np.zeros((T, d))
for s in range(n_sectors):
    cols = slice(s * m, (s + 1) * m)
    X[:, cols] = alpha * factors[:, s:s+1] + np.sqrt(1 - alpha**2) * idiosync[:, cols]

pop_spike = 1 + (m - 1) * alpha**2
print(f'Population spike eigenvalue: {pop_spike:.4f}')
print(f'Number of sectors: {n_sectors}, assets per sector: {m}')

est = RollingSpectralEstimator(window=T, step=T)
snap = est.fit_single_window(X)

print(f'\nSpectral snapshot:')
print(f'  gamma={snap.gamma:.4f}, sigma2={snap.sigma2:.4f}')
print(f'  lambda+={snap.lambda_plus:.4f}, lambda1={snap.lambda1:.4f}')
print(f'  k={snap.k} spikes (expected {n_sectors})')
print(f'  KS distance={snap.ks:.4f}')
print(f'  r_eff={snap.r_eff:.4f}')

In [ ]:
from research.analysis.plots import figure1_mp_density
fig = figure1_mp_density(snap)
plt.show()

## 5. Mathematical Flags Summary

**Flag 1**: θ_c = σ²(1+√γ) ≠ λ+ = σ²(1+√γ)². Ratio λ+/θ_c = 1+√γ > 1 always. These are distinct quantities and must not be conflated.

**Flag 2**: The BBP phase transition is proven for complex Gaussian (GUE) matrices. Application to real financial data is a heuristic. The detection threshold θ_c is approximate.

**Flag 3**: The recursive CUSUM form C(t) = max(0, C(t-1) + (x(t) - μ - δ)) is from post-1954 literature (Lorden 1971, Moustakides 1986). Page's 1954 original Rule 1 is stated in terms of min-of-cumulative-sums, not the recursive max(0,...) form.